# 7.4 Inference in first order logic: unification, chaining and resolution: practice notebook

Reason with rules that have variables. Reduce first order logic to propositional logic and see why that blows up, then lift inference with unification, prove that Colonel West is a criminal by forward chaining, backward chaining and resolution, and meet Prolog.

How to use it: run the cells from top to bottom (Shift + Enter). First read and run the examples. Then try each task in its starter cell and run the check cell under it. The full solutions are at the end, so try on your own first.

Everything here is plain Python 3. No extra packages are needed.

## Setup

In [ ]:
# A small helper. check(tests) runs each test of a task and tells you which ones pass.
# Each test is a pair: the call to make, and the answer we expect.
import json

def check(tests):
    passed = 0
    for call, expected in tests:
        try:
            got = eval(call)
        except Exception as error:
            got = "error: " + str(error)
        same = json.dumps(got, sort_keys=True, default=repr) == json.dumps(expected, sort_keys=True, default=repr)
        if same:
            passed = passed + 1
            print("PASS ", call)
        else:
            print("FAIL ", call, " expected", expected, " but got", got)
    if passed == len(tests):
        print("All", len(tests), "tests pass. Well done!")
    else:
        print(passed, "of", len(tests), "tests pass. Keep going, or look at the solution below.")

## Examples from the chapter

### Unification on the textbook pairs

In [ ]:
# Unification. A variable is a string that starts with a lower case letter, a constant starts with a capital.
# A compound term or atom is a tuple: ("Knows", "John", "x") means Knows(John, x).
def is_var(t):
    return type(t) == str and t[0].islower()

def unify(x, y, theta):
    if theta is None:
        return None
    if x == y:
        return theta
    if is_var(x):
        return unify_var(x, y, theta)
    if is_var(y):
        return unify_var(y, x, theta)
    if type(x) == tuple and type(y) == tuple and len(x) == len(y):
        for a, b in zip(x, y):
            theta = unify(a, b, theta)
        return theta
    return None

def unify_var(v, t, theta):
    if v in theta:
        return unify(theta[v], t, theta)
    if is_var(t) and t in theta:
        return unify(v, theta[t], theta)
    if occurs(v, t, theta):
        return None
    new = dict(theta)
    new[v] = t
    return new

def occurs(v, t, theta):
    if t == v:
        return True
    if is_var(t) and t in theta:
        return occurs(v, theta[t], theta)
    if type(t) == tuple:
        return any(occurs(v, a, theta) for a in t)
    return False

def subst(theta, t):
    if is_var(t) and t in theta:
        return subst(theta, theta[t])
    if type(t) == tuple:
        return tuple(subst(theta, a) for a in t)
    return t

q = ("Knows", "John", "x")
for other in [("Knows", "John", "Jane"), ("Knows", "y", "Bill"), ("Knows", "y", ("Mother", "y")), ("Knows", "x", "Elizabeth")]:
    theta = unify(q, other, {})
    if theta is None:
        print(other, "-> fail")
    else:
        print(other, "->", {v: subst(theta, v) for v in theta})

### Forward chaining on the crime example

In [ ]:
# Forward chaining on the crime example. Facts are stored by predicate name, so a premise is only
# compared with facts of the same predicate. Facts have no variables, so matching is simple.
facts = {"American": [("West",)], "Missile": [("M1",)], "Owns": [("Nono", "M1")], "Enemy": [("Nono", "America")]}
rules = [([("Missile", "x")], ("Weapon", "x")),
         ([("Missile", "x"), ("Owns", "Nono", "x")], ("Sells", "West", "x", "Nono")),
         ([("Enemy", "x", "America")], ("Hostile", "x")),
         ([("American", "x"), ("Weapon", "y"), ("Sells", "x", "y", "z"), ("Hostile", "z")], ("Criminal", "x"))]

def match(args, values, theta):
    theta = dict(theta)
    for a, v in zip(args, values):
        if a[0].islower():                    # a variable: bind it, or check its binding
            if theta.get(a, v) != v:
                return None
            theta[a] = v
        elif a != v:                          # a constant must be equal
            return None
    return theta

def all_matches(premises, theta):
    if not premises:
        return [theta]
    out = []
    first = premises[0]
    for values in facts.get(first[0], []):
        t = match(first[1:], values, theta)
        if t is not None:
            out = out + all_matches(premises[1:], t)
    return out

for rnd in [1, 2]:
    new = []
    for premises, head in rules:
        for theta in all_matches(premises, {}):
            args = tuple([theta.get(a, a) for a in head[1:]])
            if args not in facts.get(head[0], []) and (head[0], args) not in new:
                new.append((head[0], args))
    print("round", rnd, "adds", ", ".join([p + "(" + ", ".join(a) + ")" for p, a in new]))
    for p, a in new:
        facts.setdefault(p, []).append(a)

## Your turn

### Task 1: Apply a substitution

Write `subst(theta, t)`: replace every variable in t that is bound in theta, following chains of bindings (if x is bound to y and y to John, x becomes John).

Hint: A bound variable is replaced by subst of its value. A tuple is rebuilt from subst of each part. Anything else stays.

In [ ]:
# A variable is a string that starts with a lower case letter. A constant starts with a capital.
# A compound term or atom is a tuple: ("Knows", "John", "x") means Knows(John, x).
def is_var(t):
    return type(t) == str and t[0].islower()

def subst(theta, t):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([("subst({'x': 'Jane'}, ('Knows', 'John', 'x'))", ('Knows', 'John', 'Jane')), ("subst({'y': 'John', 'x': ('Mother', 'y')}, ('Knows', 'y', 'x'))", ('Knows', 'John', ('Mother', 'John'))), ("subst({}, 'x')", 'x'), ("subst({'x': 'y', 'y': 'A'}, ('P', 'x'))", ('P', 'A'))])

### Task 2: Unification with the occurs check

Write `unify(x, y, theta)` as in the textbook. Return the new dictionary of bindings, or None on failure. Use the occurs check. The starter gives is_var.

Hint: Cases: same term, x is a variable, y is a variable, two tuples of the same length (unify element by element), otherwise fail. In unify_var, follow existing bindings, then do the occurs check, then bind.

In [ ]:
# A variable is a string that starts with a lower case letter. A constant starts with a capital.
# A compound term or atom is a tuple: ("Knows", "John", "x") means Knows(John, x).
def is_var(t):
    return type(t) == str and t[0].islower()

def unify(x, y, theta):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([("unify(('Knows', 'John', 'x'), ('Knows', 'John', 'Jane'), {})", {'x': 'Jane'}), ("unify(('Knows', 'John', 'x'), ('Knows', 'y', 'Bill'), {})", {'y': 'John', 'x': 'Bill'}), ("unify(('Knows', 'John', 'x'), ('Knows', 'x', 'Elizabeth'), {})", None), ("unify(('P', 'x'), ('P', ('F', 'x')), {})", None), ("unify(('P', 'x', 'x'), ('P', 'A', 'A'), {})", {'x': 'A'})])

### Task 3: Forward chaining with variables

Write `fc_ask(facts, rules, query)`. facts is a list of ground atoms as tuples. rules is a list of `(premises, head)` pairs whose atoms may contain variables. Facts and query have no variables. Repeat rounds of forward chaining until the query is known (return True) or a round adds nothing (return False).

Hint: Write match(pattern, fact, theta) for one premise against one ground fact. all_matches tries the premises left to right, carrying theta. Each round, add every new head, then check the query.

In [ ]:
# A variable is a string that starts with a lower case letter. A constant starts with a capital.
# A compound term or atom is a tuple: ("Knows", "John", "x") means Knows(John, x).
def is_var(t):
    return type(t) == str and t[0].islower()

def fc_ask(facts, rules, query):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([("fc_ask([('American', 'West'), ('Missile', 'M1'), ('Owns', 'Nono', 'M1'), ('Enemy', 'Nono', 'America')], [([('Missile', 'x')], ('Weapon', 'x')), ([('Missile', 'x'), ('Owns', 'Nono', 'x')], ('Sells', 'West', 'x', 'Nono')), ([('Enemy', 'x', 'America')], ('Hostile', 'x')), ([('American', 'x'), ('Weapon', 'y'), ('Sells', 'x', 'y', 'z'), ('Hostile', 'z')], ('Criminal', 'x'))], ('Criminal', 'West'))", True), ("fc_ask([('American', 'West'), ('Missile', 'M1')], [([('Missile', 'x')], ('Weapon', 'x'))], ('Criminal', 'West'))", False), ("fc_ask([('Parent', 'A', 'B'), ('Parent', 'B', 'C')], [([('Parent', 'x', 'y'), ('Parent', 'y', 'z')], ('Grandparent', 'x', 'z'))], ('Grandparent', 'A', 'C'))", True), ("fc_ask([('Parent', 'A', 'B'), ('Parent', 'B', 'C')], [([('Parent', 'x', 'y'), ('Parent', 'y', 'z')], ('Grandparent', 'x', 'z'))], ('Grandparent', 'B', 'C'))", False)])

## Solutions

Try the tasks yourself first. Running a solution cell replaces your own function with the model one.

### Solution to task 1: Apply a substitution

The second test needs the chain x to Mother(y) to Mother(John).

In [ ]:
# A variable is a string that starts with a lower case letter. A constant starts with a capital.
# A compound term or atom is a tuple: ("Knows", "John", "x") means Knows(John, x).
def is_var(t):
    return type(t) == str and t[0].islower()

def subst(theta, t):
    if is_var(t) and t in theta:
        return subst(theta, theta[t])
    if type(t) == tuple:
        parts = []
        for a in t:
            parts.append(subst(theta, a))
        return tuple(parts)
    return t

In [ ]:
check([("subst({'x': 'Jane'}, ('Knows', 'John', 'x'))", ('Knows', 'John', 'Jane')), ("subst({'y': 'John', 'x': ('Mother', 'y')}, ('Knows', 'y', 'x'))", ('Knows', 'John', ('Mother', 'John'))), ("subst({}, 'x')", 'x'), ("subst({'x': 'y', 'y': 'A'}, ('P', 'x'))", ('P', 'A'))])

### Solution to task 2: Unification with the occurs check

The fourth test needs the occurs check. The last test binds x to A and then checks that the second A agrees.

In [ ]:
# A variable is a string that starts with a lower case letter. A constant starts with a capital.
# A compound term or atom is a tuple: ("Knows", "John", "x") means Knows(John, x).
def is_var(t):
    return type(t) == str and t[0].islower()

def unify(x, y, theta):
    if theta is None:
        return None
    if x == y:
        return theta
    if is_var(x):
        return unify_var(x, y, theta)
    if is_var(y):
        return unify_var(y, x, theta)
    if type(x) == tuple and type(y) == tuple and len(x) == len(y):
        for i in range(len(x)):
            theta = unify(x[i], y[i], theta)
        return theta
    return None

def unify_var(v, t, theta):
    if v in theta:
        return unify(theta[v], t, theta)
    if is_var(t) and t in theta:
        return unify(v, theta[t], theta)
    if occurs(v, t, theta):
        return None
    new = dict(theta)
    new[v] = t
    return new

def occurs(v, t, theta):
    if t == v:
        return True
    if is_var(t) and t in theta:
        return occurs(v, theta[t], theta)
    if type(t) == tuple:
        for a in t:
            if occurs(v, a, theta):
                return True
    return False

In [ ]:
check([("unify(('Knows', 'John', 'x'), ('Knows', 'John', 'Jane'), {})", {'x': 'Jane'}), ("unify(('Knows', 'John', 'x'), ('Knows', 'y', 'Bill'), {})", {'y': 'John', 'x': 'Bill'}), ("unify(('Knows', 'John', 'x'), ('Knows', 'x', 'Elizabeth'), {})", None), ("unify(('P', 'x'), ('P', ('F', 'x')), {})", None), ("unify(('P', 'x', 'x'), ('P', 'A', 'A'), {})", {'x': 'A'})])

### Solution to task 3: Forward chaining with variables

Test 1 is the crime example: two rounds. Test 4 asks for a grandparent link that does not exist.

In [ ]:
# A variable is a string that starts with a lower case letter. A constant starts with a capital.
# A compound term or atom is a tuple: ("Knows", "John", "x") means Knows(John, x).
def is_var(t):
    return type(t) == str and t[0].islower()

def match(pattern, fact, theta):
    if len(pattern) != len(fact) or pattern[0] != fact[0]:
        return None
    theta = dict(theta)
    for i in range(1, len(pattern)):
        p = pattern[i]
        f = fact[i]
        if is_var(p):
            if p in theta and theta[p] != f:
                return None
            theta[p] = f
        elif p != f:
            return None
    return theta

def all_matches(premises, facts, theta):
    if len(premises) == 0:
        return [theta]
    out = []
    for f in facts:
        t = match(premises[0], f, theta)
        if t is not None:
            out = out + all_matches(premises[1:], facts, t)
    return out

def fc_ask(facts, rules, query):
    facts = list(facts)
    if query in facts:
        return True
    while True:
        new = []
        for premises, head in rules:
            for theta in all_matches(premises, facts, {}):
                parts = [head[0]]
                for a in head[1:]:
                    parts.append(theta.get(a, a))
                fact = tuple(parts)
                if fact not in facts and fact not in new:
                    new.append(fact)
        if len(new) == 0:
            return False
        facts = facts + new
        if query in facts:
            return True

In [ ]:
check([("fc_ask([('American', 'West'), ('Missile', 'M1'), ('Owns', 'Nono', 'M1'), ('Enemy', 'Nono', 'America')], [([('Missile', 'x')], ('Weapon', 'x')), ([('Missile', 'x'), ('Owns', 'Nono', 'x')], ('Sells', 'West', 'x', 'Nono')), ([('Enemy', 'x', 'America')], ('Hostile', 'x')), ([('American', 'x'), ('Weapon', 'y'), ('Sells', 'x', 'y', 'z'), ('Hostile', 'z')], ('Criminal', 'x'))], ('Criminal', 'West'))", True), ("fc_ask([('American', 'West'), ('Missile', 'M1')], [([('Missile', 'x')], ('Weapon', 'x'))], ('Criminal', 'West'))", False), ("fc_ask([('Parent', 'A', 'B'), ('Parent', 'B', 'C')], [([('Parent', 'x', 'y'), ('Parent', 'y', 'z')], ('Grandparent', 'x', 'z'))], ('Grandparent', 'A', 'C'))", True), ("fc_ask([('Parent', 'A', 'B'), ('Parent', 'B', 'C')], [([('Parent', 'x', 'y'), ('Parent', 'y', 'z')], ('Grandparent', 'x', 'z'))], ('Grandparent', 'B', 'C'))", False)])